# Lesson 64 · Training and evaluation — pushing the loss down yourself: the train/eval loop, the confusion matrix and diagnosing overfitting

**Goal**: implement `train_epoch` / `eval_accuracy`, and connect training, validation and the confusion matrix into a loop that can diagnose problems.

> Passing isn't just about accuracy: look at the train/val trends and the signs of overfitting (train going down, val going up) together. Only once this is wired to L62/L63 is it truly end to end.

🔗 **Aurora connection**:
- the mel features come from `aurora.audio.mfcc` (STFT → mel filterbank → log); see `src/aurora/audio/`
- this lesson uses fake data + a small MLP to demonstrate the skeleton of the training loop; swap in the real `KeywordCNN` (L63) and `KWSDataset` (L62) and it becomes the Aurora project's first end-to-end trained model, whose weights the later inference module loads directly


← **Previous lesson** · [L63 · An audio classifier](L63_kws_model.ipynb)

> Last lesson covered **an audio classifier**: a CNN on mel features, defining the network for Speech Commands.
> This lesson explores **training and evaluation**.

## This lesson's story: why isn't "watching the loss go down" enough — why do we need a confusion matrix too?

A falling loss is only the surface. The real question is: what is the model actually learning, and on which classes is it making mistakes?

**The problem**: if "yes" makes up 80% of the dataset, a model that always guesses "yes" scores 80% accuracy. But that isn't recognition at all — it's an illusion created by **class imbalance**.

**Three diagnostic tools**:

| Tool | What it shows | What problem it reveals |
|---|---|---|
| training loss | whether the model is learning | overfitting / underfitting |
| validation accuracy | generalization | an overall verdict, but it hides imbalance |
| confusion matrix | the number of samples for each (true, predicted) pair | which classes get confused with each other |

**Cross-entropy loss** (`nn.CrossEntropyLoss`) takes logits (the raw outputs before softmax) and internally computes the softmax and then the log loss:
```
CE = -log(softmax(logit)[y])    ← the negative log-probability of the correct class
initial value ≈ log(num_classes) = log(10) ≈ 2.3   ← the expected loss of random guessing
```

**The training loop = two functions**:
```
train_epoch(model, loader, optimizer, criterion) → avg_loss
eval_accuracy(model, loader)                      → accuracy %
```

Your task in this section: implement these two functions so that within 5 epochs the training loop shows the loss falling and the accuracy rising.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

## 0.5 Softmax: turning scores into comparable probabilities

Before we get to the loss function, let's understand softmax, the "probability converter".

### The question: why convert the model's outputs into probabilities?

When you train a classifier, the numbers the model's last layer outputs are called **logits** (raw scores). For example:
```
sample x through the model → [2.0, 0.5, -1.0]  (raw scores for the classes "yes", "no", "stop")
```

These three numbers have no clear meaning — what does 2.0 mean? Is being four times 0.5 "four times as certain"? To make the model's outputs **easy to understand and interpret**, we need to turn these scores into "probabilities" (between 0 and 1, adding up to 1). Then 0.7 means "a 70% chance it's yes".

### What is softmax, and why does it use the exponential?

**The softmax formula**:
```
softmax(logit)[j] = exp(logit[j]) / sum_k exp(logit[k])
```

Breaking the formula down:
- **numerator** `exp(logit[j])`: exponentiate the j-th score
- **denominator** `sum_k exp(logit[k])`: the sum of the exponentials of all scores (for normalization)
- **result**: a number between 0 and 1, and all classes add up to 1

**Why use the exponential exp() rather than just logit / sum(logit)?**

Let's compare. Suppose logit = [2.0, 0.5, -1.0]:

1. **Plain normalization** (logit / sum(logit)):
   - denominator = 2.0 + 0.5 + (-1.0) = 1.5
   - result = [2.0/1.5, 0.5/1.5, -1.0/1.5] = [1.33, 0.33, -0.67]
   - ❌ a **negative number** appears! That's not a probability (probabilities can't be negative)

2. **Softmax** (exponential normalization):
   - exp([2.0, 0.5, -1.0]) ≈ [7.39, 1.65, 0.37] (all positive)
   - denominator = 7.39 + 1.65 + 0.37 = 9.41
   - result ≈ [0.785, 0.175, 0.040] (all positive, adding up to 1)
   - ✅ a perfect probability distribution

**What's clever about the exponential**:
- `exp(x)` returns a **positive number** for every x (always > 0), so the result is always a valid probability
- **It amplifies large values**: a large logit (like 2.0) becomes relatively larger after exp (exp(2.0) ≈ 7.39), boosting its weight
- **It shrinks small values**: a small logit (like -1.0) becomes very small after exp (exp(-1.0) ≈ 0.37) and barely affects the result

That way, a relatively large score in the logits is read as "this class is more likely" — exactly the behavior a classifier wants.

### Its role in cross-entropy

After the logits are turned into probabilities by softmax, the loss function takes **the negative log of the correct class's probability**:
```
CE = -log(softmax(logit)[correct class])
```

So:
- the **more certain** the model (softmax closer to 1) → log closer to 0 → smaller loss ✅
- the **less certain** the model (softmax closer to 0) → log closer to -∞ → huge loss ❌

That's the whole picture.

## 1. Cross-entropy loss and the Adam optimizer

**Cross-entropy loss** (`nn.CrossEntropyLoss`) takes raw logits (shape `[B, C]`) and integer labels (shape `[B]`), and internally applies softmax followed by the negative log-likelihood:

```
loss = -log( exp(logit[y]) / sum_j exp(logit[j]) )
```

Don't add a softmax by hand at the end of the model — `CrossEntropyLoss` already includes it.

**The Adam optimizer** combines momentum (the first moment) with an adaptive learning rate (the second moment). Keyword spotting (KWS) tasks usually start from a learning rate of `lr=1e-3`.

```python
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
```

In [ ]:
# Demo: what softmax does
import torch.nn.functional as F

# logits for three samples (raw model outputs)
logits_demo = torch.tensor([
    [2.0, 0.5, -1.0],   # sample 1: confident it's class 0
    [0.3, 0.3, 0.3],    # sample 2: completely unsure
    [-2.0, -0.5, 1.5]   # sample 3: confident it's class 2
], dtype=torch.float32)

print("Raw logits (model output):")
print(logits_demo)
print()

# compute softmax by hand (using the first sample)
sample_0_logit = logits_demo[0]  # [2.0, 0.5, -1.0]
exp_vals = torch.exp(sample_0_logit)
softmax_manual = exp_vals / exp_vals.sum()
print(f"sample 0 logit = {sample_0_logit.tolist()}")
print(f"  after exp = {exp_vals.tolist()}")
print(f"  after normalizing (softmax) = {softmax_manual.tolist()}")
print(f"  check the sum = {softmax_manual.sum().item():.6f}")
print()

# PyTorch's built-in softmax (should give the same result)
probs = F.softmax(logits_demo, dim=1)
print("Probability distributions of all samples after softmax:")
print(probs)
print()

# interpret these probabilities
for i, prob_i in enumerate(probs):
    most_likely = prob_i.argmax().item()
    confidence = prob_i[most_likely].item()
    print(f"sample {i}: most likely class {most_likely}, confidence {confidence:.1%}")


In [ ]:
# Demo: how CrossEntropyLoss behaves
criterion_demo = nn.CrossEntropyLoss()

# 3 samples, 4 classes
logits = torch.tensor([[2.0, 0.5, 0.1, 0.1],   # confidently predicts class 0
                        [0.1, 0.1, 2.0, 0.1],   # confidently predicts class 2
                        [0.5, 0.5, 0.5, 0.5]])  # completely unsure
labels = torch.tensor([0, 2, 1])  # true labels

loss_val = criterion_demo(logits, labels)
print(f'loss: {loss_val.item():.4f}')
print('the first two samples are predicted correctly and confidently → small loss; the third is unsure and wrong → large loss')

In [ ]:
# Going deeper: why is the loss at initialization ≈ log(num_classes)?
import torch
import torch.nn as nn

print("=" * 60)
print("Is the cross-entropy loss of an initialized model = log(number of classes)?")
print("=" * 60)

num_classes = 10
criterion = nn.CrossEntropyLoss()

# Scenario 1: completely random initialization (logits are all small and close together)
torch.manual_seed(42)
random_logits = torch.randn(1000, num_classes) * 0.01  # N(0, 0.01), very small
random_labels = torch.randint(0, num_classes, (1000,))
loss_random = criterion(random_logits, random_labels)

print(f"\nScenario 1: random initialization (Gaussian N(0, 0.01))")
print(f"  average loss over 1000 samples = {loss_random.item():.4f}")
print(f"  log(10) = {torch.log(torch.tensor(10.0)).item():.4f}")
print(f"  ✅ essentially equal!")

# Scenario 2: step-by-step analysis of why
print(f"\nScenario 2: why is it log(num_classes)?")
print(f"  when all the logits are close to 0:")
print(f"    exp(0) = 1")
print(f"    softmax[j] = 1 / (1 + 1 + ... + 1)  (ten 1s)")
print(f"             = 1 / 10 = 0.1")
print(f"    CE = -log(0.1) = -log(1/10) = log(10) = {torch.log(torch.tensor(10.0)).item():.4f}")

# Scenario 3: what "random guessing" means
print(f"\nScenario 3: compare against the true labels")
correct_labels = torch.zeros(1000, dtype=torch.long)  # every label is class 0
loss_correct = criterion(random_logits, correct_labels)
print(f"  even with random initialization, if every label is 0 the CE is still about {loss_correct.item():.4f}")
print(f"  because a randomly initialized model gives every class equal probability, so it gives class 0 only 1/10 too")

# Scenario 4: training improves it
print(f"\nScenario 4: training pushes the loss down")
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, num_classes)
)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# a super simple dataset: 128 samples, 100 features, random labels
X_simple = torch.randn(128, 100)
y_simple = torch.randint(0, num_classes, (128,))

loss_before = criterion(model(X_simple), y_simple).item()
print(f"  loss before training = {loss_before:.4f}")

# train for 50 steps
for _ in range(50):
    optimizer.zero_grad()
    logits = model(X_simple)
    loss = criterion(logits, y_simple)
    loss.backward()
    optimizer.step()

loss_after = criterion(model(X_simple), y_simple).item()
print(f"  loss after 50 training steps = {loss_after:.4f}")
print(f"  ✅ the loss went from {loss_before:.4f} → {loss_after:.4f}, pushed down!")


## 2. Per-class accuracy

Overall accuracy `correct / total` hides class imbalance. Per-class accuracy counts each keyword class separately:

```
accuracy_c = (number of samples predicted as c and truly c) / (total number of samples truly c)
```

If a class makes up 1% of the dataset, 99% overall accuracy can be achieved by ignoring it completely; per-class accuracy exposes the problem.

In [ ]:
# Demo: computing per-class accuracy by hand
y_true = torch.tensor([0, 0, 1, 1, 2, 2, 2])
y_pred = torch.tensor([0, 1, 1, 1, 0, 2, 2])  # one class-0 mistake, one class-2 mistake

num_classes = 3
for c in range(num_classes):
    mask = (y_true == c)
    acc_c = (y_pred[mask] == c).float().mean().item()
    print(f'  class {c}: {acc_c:.2%}')

overall = (y_pred == y_true).float().mean().item()
print(f'overall accuracy: {overall:.2%}')

## 3. The confusion matrix

The confusion matrix (CM) `C` has shape `[num_classes, num_classes]`:

```
C[i][j] = number of samples whose true label is i and prediction is j
```

- **diagonal** `C[i][i]`: correct predictions
- **off-diagonal** `C[i][j]` (i≠j): wrong predictions (class i mistaken for class j)

When visualized as a heatmap, a darker color means more samples in that cell; ideally only the diagonal of the heatmap is dark.

In [ ]:
# Demo: computing and visualizing a confusion matrix with sklearn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true_np = y_true.numpy()
y_pred_np = y_pred.numpy()
class_names = ['yes', 'no', 'stop']

cm = confusion_matrix(y_true_np, y_pred_np)
print('Confusion matrix:')
print(cm)

fig, ax = plt.subplots(figsize=(4, 3))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion matrix example')
plt.tight_layout()
plt.show()

In [ ]:
# How to read the colors of the confusion-matrix heatmap
print("\nReading the heatmap colors:")
print(f"  darker (deeper blue) → more samples in this cell")
print(f"  lighter (close to white) → fewer samples in this cell")
print()
print(f"The ideal case:")
print(f"  ✅ the diagonal C[i][i] is very dark (many correct predictions)")
print(f"  ✅ the off-diagonal cells are all very light (few wrong predictions)")
print()
print(f"A heatmap like that looks like this:")
print(f"  a dark blue diagonal running through the matrix")
print(f"  white or pale blue everywhere around it")


## 3.5 model.train() and model.eval(): a neural network's two working modes

You may have noticed two mysterious commands in the training loop:
```python
model.train()   # when training
model.eval()    # when evaluating
```

What do these two lines actually do, and why must they be kept apart? Let's start with two special layers.

### Dropout: "randomly dropping" during training

**The problem**: when a neural network trains very well on the training set, it sometimes "over-relies" on particular neurons or feature combinations. On a different dataset it falls apart — this is called **overfitting**.

**The idea of Dropout**: during training, randomly switch off some neurons (with probability `p`, say 50%), so the network is forced to learn "several routes" to get the job done. An analogy:

> Suppose you always take notes with the same blue pen. One day that pen goes missing and you're lost. But if you practice by switching pens at random (sometimes blue, sometimes black, sometimes no pen at all), you learn to understand the material in more than one way. On the day of the real exam, you'll do well whatever pen you have.

Code example:
```python
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(100, 50)
        self.dropout = nn.Dropout(p=0.5)  # 50% chance of dropping
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.dropout(x)      # ← random dropping happens here
        x = self.fc2(x)
        return x

model = Model()

# training: Dropout drops at random
model.train()
for _ in range(100):
    logits = model(x_train)  # each forward pass drops different neurons
    # ...

# evaluation: Dropout keeps every neuron
model.eval()
with torch.no_grad():
    logits = model(x_val)    # every neuron is used (no dropping, no rescaling)
```

**The difference between train and eval mode**:
- `model.train()`: Dropout randomly drops neurons (and their weights) with probability `p`, and scales the surviving outputs up by `1/(1-p)` so the expected value stays the same
- `model.eval()`: Dropout keeps everything and passes the values through unchanged

### BatchNorm: "normalizing the data" during training

**The problem**: the distribution of a hidden layer's inputs keeps shifting as the weights are updated (called internal covariate shift), which makes training unstable.

**The idea of BatchNorm**: each hidden layer **normalizes** the current batch's data (subtract the mean, divide by the std), so the data always stays in a sensible range.

```python
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(100, 50)
        self.bn1 = nn.BatchNorm1d(50)  # BN layer
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)   # ← mean/std computed here
        x = torch.relu(x)
        x = self.fc2(x)
        return x
```

**The core difference between train and eval mode**:

| Mode | How it normalizes | Why |
|---|---|---|
| **training** (`model.train()`) | with the current batch's mean/std | the batch reflects the data as it is right now; updating the weights needs "just right" normalization |
| **evaluation** (`model.eval()`) | with the global running mean/std | an evaluation set may have only a few dozen samples and can't represent the whole distribution; use the global statistics accumulated during training instead |

**An analogy**:
> A water thermometer: during training, every batch is a fresh glass of water, and its temperature defines "normal" for that glass. At evaluation time you don't have enough water samples, so you use the average temperature accumulated in the past.

### Why both model.eval() and torch.no_grad()?

You may have seen code like this:
```python
model.eval()
with torch.no_grad():
    predictions = model(x_test)
```

Are both lines needed?

- **`model.eval()`**: turns Dropout off (keeps every neuron) and tells BatchNorm to use the running statistics. But **gradient computation is still on**.
- **`torch.no_grad()`**: turns off autograd, saving memory and compute time.

So each has its own job:
- `eval()` changes the network's **behavior** (how Dropout and BatchNorm work)
- `no_grad()` changes PyTorch's **computation mode** (no computation graph is built)

If you use only `model.eval()` without `no_grad()`, the code still runs but wastes memory; if you use only `no_grad()` without `eval()`, Dropout keeps dropping at random and your evaluation results are wrong.

## 4. ✏️ Implement `train_epoch(model, loader, optimizer, criterion)`

**Five steps (repeated for every batch)**:

| Step | Operation | Note |
|---|---|---|
| 1 | `model.train()` | switch dropout/BN into training mode (at the start of the epoch) |
| 2 | `optimizer.zero_grad()` | clear the gradients; required for every batch |
| 3 | `loss = criterion(model(x), y)` | forward pass + compute the loss |
| 4 | `loss.backward()` | backpropagation |
| 5 | `optimizer.step()` | update the parameters |

**Return value**: `avg_loss: float` = the average loss over all batches

**Acceptance criteria**:
- the function returns a float, not a Tensor
- the loss should fall over repeated calls (for a sensible lr and data)

In [ ]:
# Why is zero_grad() required? A cautionary example
print("=" * 60)
print("Demo: what happens if you forget zero_grad()")
print("=" * 60)

torch.manual_seed(0)
model_bad = nn.Linear(3, 1)  # a simple model, 3 inputs, 1 output
optimizer_bad = torch.optim.SGD(model_bad.parameters(), lr=0.1)
criterion_bad = nn.MSELoss()

# deliberately skip zero_grad()
x_demo = torch.tensor([[1.0, 2.0, 3.0], [2.0, 3.0, 4.0]])
y_demo = torch.tensor([[1.0], [2.0]])

print("\n[Wrong] not clearing the gradients:")
for i in range(3):
    out = model_bad(x_demo)
    loss = criterion_bad(out, y_demo)
    loss.backward()
    # ❌ no optimizer.zero_grad()
    optimizer_bad.step()
    print(f"  Epoch {i+1}: loss={loss.item():.4f}, weight={model_bad.weight.data[0, 0].item():.4f}, "
          f"grad={model_bad.weight.grad[0, 0].item():.4f}")

print("\nAnalysis: the gradients are accumulating! Round 1 uses g1, round 2 uses g2+g1, round 3 uses g3+g2+g1")
print("So step() updates with the wrong gradient direction and the weights jump around unstably.")

print("\n[Right] clear the gradients every round:")
torch.manual_seed(0)
model_good = nn.Linear(3, 1)
optimizer_good = torch.optim.SGD(model_good.parameters(), lr=0.1)
criterion_good = nn.MSELoss()

for i in range(3):
    optimizer_good.zero_grad()  # ✅ clear first
    out = model_good(x_demo)
    loss = criterion_good(out, y_demo)
    loss.backward()
    optimizer_good.step()
    print(f"  Epoch {i+1}: loss={loss.item():.4f}, weight={model_good.weight.data[0, 0].item():.4f}, "
          f"grad={model_good.weight.grad[0, 0].item():.4f}")

print("\n✅ Done right, each round's gradient is computed independently and the weight updates are stable.")

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    """Train for one epoch and return the average loss."""
    # ✏️ TODO: model.train()
    # ✏️ TODO: for x, y in loader:
    #              optimizer.zero_grad()
    #              out = model(x)
    #              loss = criterion(out, y)
    #              loss.backward()
    #              optimizer.step()
    #              accumulate loss.item()
    # ✏️ TODO: return avg_loss
    raise NotImplementedError

In [ ]:
# Visual check of train_epoch
# a quick smoke test with a tiny fake model + fake loader
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
_x = torch.randn(16, 1, 40, 32)   # 16 samples, single-channel mel images
_y = torch.randint(0, 4, (16,))   # random labels from 4 classes
_loader = DataLoader(TensorDataset(_x, _y), batch_size=8)

_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(40*32, 4)
)
_opt = torch.optim.Adam(_model.parameters(), lr=1e-3)
_crit = nn.CrossEntropyLoss()

try:
    _loss = train_epoch(_model, _loader, _opt, _crit)
    assert isinstance(_loss, float), 'the return value should be a float'
    assert _loss > 0, 'the loss should be greater than 0'
    print(f'✅ train_epoch passed, avg_loss = {_loss:.4f}')

    # stronger check: confirm the weights really get updated (the epoch-2 loss should be lower than epoch 1)
    _loss_e2 = train_epoch(_model, _loader, _opt, _crit)
    assert _loss_e2 < _loss, (
        f'the epoch-2 loss ({_loss_e2:.4f}) should be lower than epoch 1 ({_loss:.4f}); '
        'check that optimizer.step() is called correctly and that zero_grad() runs before every batch'
    )
    print(f'✅ weight-update check passed: epoch1={_loss:.4f} → epoch2={_loss_e2:.4f}')
except (NotImplementedError, TypeError):
    print('⚠️ train_epoch is not implemented yet — finish the TODO in section 4 and re-run this cell')

## 4.5 argmax(dim=1): from scores to class predictions

To evaluate accuracy, we need to turn the model's output (a score for each class) into "which class does this sample belong to". That's what `argmax` does.

### The shape of the model's output

Let's start with a concrete example. Suppose you have:
- **10 samples** (a batch)
- **4 classes** (say "yes", "no", "stop", "go")

Passing these 10 samples through the model gives a tensor of shape `(10, 4)` (10 rows, 4 columns):

```
sample\class  yes    no     stop   go
0         [2.5,  0.1,  -1.2,  0.3]   ← sample 0's scores for the 4 classes
1         [0.2,  3.1,   0.5,  1.0]   ← sample 1's scores for the 4 classes
2         [-0.1, 1.5,   2.8,  0.2]   ← sample 2's scores for the 4 classes
...
9         [1.0,  0.5,   0.5,  1.5]   ← sample 9's scores for the 4 classes
```

Each **row** (the i-th sample) is a vector of length 4: this sample's "scores" (logits) for the 4 classes.

### What argmax(dim=1) means

`argmax(dim=1)` takes the **index** of the maximum along dimension 1 (across the columns):

```python
logits = torch.tensor([
    [2.5, 0.1, -1.2, 0.3],   # sample 0: max is 2.5 (at index 0) → predict class 0
    [0.2, 3.1, 0.5, 1.0],    # sample 1: max is 3.1 (at index 1) → predict class 1
    [-0.1, 1.5, 2.8, 0.2],   # sample 2: max is 2.8 (at index 2) → predict class 2
])

preds = logits.argmax(dim=1)
# result: tensor([0, 1, 2])
```

**Why do it this way?** Because the larger the logit, the larger the corresponding probability after softmax (the exponential "amplifies" it). So the highest-scoring class is the one the model is most confident about.

### Compare with the other dimension

What happens if you use `dim=0` by mistake?

```python
preds_wrong = logits.argmax(dim=0)
# result: tensor([0, 1, 2, 1])
# this says: the highest score for class 0 came from sample 0, the highest for class 1 from sample 1, ...
# ❌ not what we want! We want "which class each sample belongs to", not "which sample each class comes from".
```

In [ ]:
# Demo: using argmax(dim=1)
print("=" * 60)
print("argmax(dim=1): from scores to predicted classes")
print("=" * 60)

logits_example = torch.tensor([
    [2.5, 0.1, -1.2, 0.3],    # sample 0
    [0.2, 3.1, 0.5, 1.0],     # sample 1
    [-0.1, 1.5, 2.8, 0.2],    # sample 2
    [1.0, 1.0, 1.0, 2.5],     # sample 3
], dtype=torch.float32)

class_names = ['yes', 'no', 'stop', 'go']
print(f"\nLogits tensor shape: {logits_example.shape}  (4 samples, 4 classes)")
print(f"Logits contents:")
print(logits_example)

# index of the maximum along dim=1
preds = logits_example.argmax(dim=1)
print(f"\nargmax(dim=1) result: {preds.tolist()}")

# explain each prediction
print(f"\nOne by one:")
for i, (logit_row, pred_idx) in enumerate(zip(logits_example, preds)):
    max_score = logit_row[pred_idx].item()
    pred_class = class_names[pred_idx]
    print(f"  sample {i}: scores={logit_row.tolist()}")
    print(f"         max {max_score:.2f} at index {pred_idx} → predict '{pred_class}'")

# wrong example: using dim=0
print(f"\n[Wrong] if you use argmax(dim=0):")
preds_wrong = logits_example.argmax(dim=0)
print(f"result: {preds_wrong.tolist()}")
print(f"meaning: \"the highest score for class 0 came from sample {preds_wrong[0]}, the highest for class 1 from sample {preds_wrong[1]}\"")
print(f"❌ not what we want!")


## 5. ✏️ Implement `eval_accuracy(model, loader)`

**Three steps**:

| Step | Operation | Key detail |
|---|---|---|
| 1 | `model.eval()` + `torch.no_grad():` | turn off dropout, disable gradient computation (saves memory) |
| 2 | `preds = model(x).argmax(dim=1)` | index of the largest logit = predicted class |
| 3 | `correct += (preds == y).sum()` | accumulate the number correct, divide by total at the end |

**Return value**: `accuracy: float` ∈ [0.0, 1.0]

**Acceptance criteria**:
- returns a float (not a Tensor)
- ≈ 1/num_classes at random initialization (random guessing)
- should beat the random baseline after training

In [ ]:
def eval_accuracy(model, loader):
    """Compute the overall accuracy on the validation set."""
    # ✏️ TODO: model.eval()
    # ✏️ TODO: with torch.no_grad():
    #              for x, y in loader:
    #                  preds = model(x).argmax(dim=1)
    #                  accumulate correct / total
    # ✏️ TODO: return correct / total
    raise NotImplementedError

In [ ]:
# Visual check of eval_accuracy
try:
    _acc_before = eval_accuracy(_model, _loader)
    assert 0.0 <= _acc_before <= 1.0, 'accuracy should be in [0, 1]'
    print(f'✅ eval_accuracy passed, accuracy = {_acc_before:.2%}')

    # stronger check: a boundary case — a perfectly predicting model should return accuracy 1.0
    _bnd_x = torch.zeros(4, 1, 40, 32)
    _bnd_y = torch.tensor([0, 1, 2, 3])
    _bnd_loader = DataLoader(TensorDataset(_bnd_x, _bnd_y), batch_size=4)

    class _PerfectClassifier(nn.Module):
        """A placeholder model that is always right: sample i is predicted as class i."""
        def forward(self, x):
            # 4 samples, 4 classes; torch.eye(4) makes each sample's correct-class logit the largest
            return torch.eye(4)

    _acc_perfect = eval_accuracy(_PerfectClassifier(), _bnd_loader)
    assert abs(_acc_perfect - 1.0) < 1e-6, (
        f'a perfect predictor should return 1.0, but got {_acc_perfect:.4f}; '
        'check the counting logic of (preds == y).sum()'
    )
    print(f'✅ boundary check passed: perfect-predictor accuracy = {_acc_perfect:.2%}')
except (NotImplementedError, TypeError):
    print('⚠️ eval_accuracy is not implemented yet — finish the TODO in section 5 and re-run this cell')

## 5.5 Overfitting and the generalization gap

You've probably heard the word "overfitting", but what exactly does it mean? And why do we need separate training and validation sets?

### What is overfitting?

A story to make it concrete:

> Two students are preparing for their final exams.
>
> **David** (overfitting): he got hold of 5 past papers and memorized them inside out. Give him those 5 papers with a number or two changed and he'd fly through them. But the exam questions are brand new (even though they test the same material), and he's lost — he scores 60.
>
> **Sarah** (generalization): she understood the concepts deeply and practiced all kinds of variations, so she knows "why" rather than just "how". In the exam she hasn't seen these exact questions, but she can apply what she knows flexibly — she scores 85.
>
> Outside the exam hall a friend asks David: "But you got the past papers all right, didn't you?" David says: "Sure, 100%. I just can't do new questions." — That's **overfitting**: perfect on the training data (the past papers), poor on new data (the new questions).

### How do you diagnose overfitting? You need two curves

You need to record separately:
1. **The training loss curve**: the model's loss on the data it's trained on
2. **The validation accuracy curve**: the model's accuracy on a separate validation set

**The symptom of overfitting** (the "gap" between the two curves):
```
training loss:       2.3 → 1.5 → 0.8 → 0.5 → 0.3 → 0.2 (keeps falling)
validation accuracy: 25% → 50% → 70% → 75% → 76% → 76% (levels off)
                                           ↑ stops rising here
```

- Early on (epochs 1–3): training and validation both improve ✅
- Later (epochs 4–5): training keeps improving but validation stalls ⚠️ **overfitting begins**
  - the model is "memorizing" details of the training data
  - those details don't help on new data

### Why use a separate validation set?

Imagine what happens if the training set = the validation set (the same data):

```
the model has seen this data and can memorize it completely:
  sample A: input x_A → label y_A ✓
  sample B: input x_B → label y_B ✓
  ...

validating on the same data:
  sample A: input x_A → model outputs y_A (because it memorized it) ✓
  sample B: input x_B → model outputs y_B (because it memorized it) ✓
  ...

result: training accuracy = validation accuracy = 100%

but this only proves the model can "memorize", not that it can "understand" and "generalize"!
```

**Why a separate validation set matters**:
- the model has **never seen** the validation data (it wasn't used for training)
- the model can't "memorize" it and has to rely on patterns it really learned
- only then does the "gap" between the two curves truly reflect how much it overfits

### Defining the "generalization gap"

**Generalization gap** = the difference between training performance and validation performance (which way round depends on which metric you look at)

```
generalization gap = validation loss - training loss  (ideally < 0.1)
       or
generalization gap = training accuracy - validation accuracy  (ideally close to 0)
```

- gap = 0: perfect generalization (a bit unrealistic)
- gap < 0.1: very good generalization
- gap > 0.2: clear overfitting; time to act (e.g. Dropout, L2 regularization, more training data)

### Next: how do you prevent or diagnose overfitting?

In this lesson (L64) you've learned to **observe** overfitting with a **separate validation set**. The next lesson (L65) teaches you to use **visualization tools** (gradient histograms, weight distributions and so on) to **diagnose** its causes, and techniques such as **early stopping** to **hold it back**.

## 6. Parameter experiment: training curves + confusion matrix

**Experiment parameters**:
- `num_epochs = 5`: watch the shape of the loss curve as it falls from its initial value (≈log(num_classes)≈1.39)
- `lr = 1e-3` (the Adam default): try `1e-4` to compare convergence speed
- `batch_size = 16`: small-batch training, so the loss curve will wobble

**What to expect**:
- the loss curve falls fastest in the first 2 epochs, then flattens out
- the confusion-matrix diagonal gradually darkens and the off-diagonal cells fade
- if two classes are badly confused (a dark off-diagonal cell), their mel features are similar, and you may need more training data or a deeper network

In [ ]:
# Parameter experiment: 5-epoch training curves + confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# re-initialize the model (fake data for the demo; swap in the real KeywordCNN + DataLoader)
torch.manual_seed(0)
exp_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(40*32, 64),
    nn.ReLU(),
    nn.Linear(64, 4)
)
exp_opt = torch.optim.Adam(exp_model.parameters(), lr=1e-3)
exp_crit = nn.CrossEntropyLoss()

# fake-data loader (in real use, replace with a DataLoader over SpeechCommandsDataset)
torch.manual_seed(1)
_x_big = torch.randn(128, 1, 40, 32)
_y_big = torch.randint(0, 4, (128,))
# ✅ the right way: use random_split to cut the dataset into separate training and validation sets
# if val_loader and train_loader use the same data, "validation accuracy" equals training accuracy,
# the model can never show a generalization gap, and diagnosing overfitting becomes impossible.
from torch.utils.data import random_split
_full_ds = TensorDataset(_x_big, _y_big)
_train_ds, _val_ds = random_split(_full_ds, [100, 28],
                                  generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(_train_ds, batch_size=16, shuffle=True)
val_loader   = DataLoader(_val_ds,   batch_size=16)

num_epochs = 5
train_losses = []
val_accs = []

try:
    for epoch in range(1, num_epochs + 1):
        loss = train_epoch(exp_model, train_loader, exp_opt, exp_crit)
        acc  = eval_accuracy(exp_model, val_loader)
        train_losses.append(loss)
        val_accs.append(acc)
        print(f'Epoch {epoch}/{num_epochs}  loss={loss:.4f}  acc={acc:.2%}')
except (NotImplementedError, TypeError):
    print('⚠️ train_epoch / eval_accuracy are not implemented yet — finish the TODOs in sections 4 and 5 and re-run this cell')

if train_losses:
    # plot the training curves
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].plot(range(1, num_epochs+1), train_losses, marker='o', color='steelblue')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Cross-Entropy Loss')
    axes[0].set_title('Training loss curve')
    axes[0].grid(alpha=0.3)

    axes[1].plot(range(1, num_epochs+1), val_accs, marker='s', color='tomato')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Validation accuracy curve')
    axes[1].set_ylim(0, 1)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # confusion matrix
    exp_model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            all_preds.append(exp_model(xb).argmax(dim=1))
            all_labels.append(yb)
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    cm = confusion_matrix(all_labels, all_preds)
    class_names = ['yes', 'no', 'stop', 'go']

    fig2, ax2 = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
    disp.plot(ax=ax2, colorbar=True, cmap='Blues')
    ax2.set_title('Validation confusion matrix (after 5 epochs)')
    plt.tight_layout()
    plt.show()

## Wrap-up

This section implemented `train_epoch` (returning `avg_loss: float`) and `eval_accuracy` (returning `accuracy: float`), and used fake data (`torch.randn` + a small MLP) to run the complete skeleton of a supervised training loop. Swap the model for L63's `KeywordCNN` and the data for L62's `KWSDataset` (whose mel features come from the hand-written STFT → mel filterbank pipeline in `aurora.audio` (`src/aurora/audio/`)), and you have the Aurora project's first end-to-end trained neural network. The next lesson (L65) visualizes the training process in depth: plotting loss/accuracy curves live, monitoring gradient norms and weight-distribution histograms, to help diagnose common training problems such as overfitting and vanishing gradients.

> **On the scope of overfitting diagnosis**: this lesson (L64) sets up the basic structure for observing overfitting with a separate validation set (the `random_split` in the section 6 parameter experiment) — training loss keeps falling while validation accuracy levels off. L65 provides systematic, quantitative tools (an early-stopping criterion, gradient-norm monitoring and so on); taken together, the two lessons cover the full path from "introducing overfitting" to "diagnosing it in depth".

## ✏️ Whiteboard challenge: work out the train/eval loop by hand (target: 10 minutes)

Cover the screen and derive on paper:

**Question 1**: what is the expected `nn.CrossEntropyLoss` of a randomly initialized model? (10-class classification)
(log(10)≈2.303: the softmax output is ≈ uniform, each class gets probability ≈0.1, and -log(0.1)=2.303)

**Question 2**: why must `eval_accuracy` use `torch.no_grad()` while `train_epoch` doesn't?
(inference needs no gradients, so no_grad cuts memory use and computation; training needs gradients for backward)

**Question 3**: which layers are affected by `model.train()` and `model.eval()`?
(Dropout: drops at random in train mode, keeps everything in eval mode; BatchNorm: batch statistics in train mode, running mean/var in eval mode)

**Question 4**: what does `C[i][j]` in a confusion matrix mean? What does the diagonal represent?
(C[i][j] = the number of samples of true class i predicted as j; the diagonal C[i][i] = the number of correct predictions; ideally only the diagonal is non-zero)

**Question 5**: if `eval_accuracy` returned a Tensor instead of a float, what would happen to a later `print(f"acc={acc:.4f}")`?
(a Tensor supports formatted printing, but in other contexts it can trigger gradient tracking or device errors; use .item() to convert to a float)

Once you've worked them out, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import torch, torch.nn as nn

# Q1: CE loss of a random model ≈ log(num_classes)
criterion_q = nn.CrossEntropyLoss()
n_classes = 10
logits_q = torch.zeros(100, n_classes)   # uniform logits
labels_q = torch.randint(0, n_classes, (100,))
loss_q = criterion_q(logits_q, labels_q)
expected = torch.log(torch.tensor(float(n_classes)))
assert abs(loss_q.item() - expected.item()) < 0.01, f"loss={loss_q.item():.4f}"
print(f"Q1 ✅  with uniform logits CE={loss_q.item():.4f} ≈ log({n_classes})={expected.item():.4f}")

# Q2: no_grad saves memory
x_q = torch.randn(4, 1, 40, 81, requires_grad=True)
with torch.no_grad():
    y_q = x_q * 2
assert y_q.grad_fn is None
print(f"Q2 ✅  inside no_grad, grad_fn={y_q.grad_fn} (no computation graph, saves memory)")

# Q3: how train/eval mode affects Dropout
import torch.nn.functional as F
torch.manual_seed(0)
dp = nn.Dropout(p=0.5)
x_test = torch.ones(100)
dp.train(); out_train = dp(x_test)
dp.eval();  out_eval  = dp(x_test)
train_zeros = (out_train == 0).sum().item()
eval_zeros  = (out_eval  == 0).sum().item()
assert eval_zeros == 0, "Dropout should not drop anything in eval mode"
assert train_zeros > 0, "Dropout should drop some values in train mode"
print(f"Q3 ✅  Dropout: train mode dropped {train_zeros}/100, eval mode dropped {eval_zeros}/100")

# Q4: the confusion-matrix diagonal
cm_q = torch.zeros(3, 3, dtype=torch.long)
preds_q = torch.tensor([0, 0, 1, 1, 2, 2])
labels_q2 = torch.tensor([0, 1, 1, 2, 2, 0])
for p, l in zip(preds_q, labels_q2):
    cm_q[l, p] += 1
diagonal_sum = cm_q.diagonal().sum().item()
total_q2 = len(preds_q)
print(f"Q4 ✅  confusion-matrix diagonal sum = {diagonal_sum}: {diagonal_sum}/{total_q2} correct")
print(f"      C[i][j] = true i predicted as j: {cm_q.tolist()}")

# Q5: Tensor vs float return value
t_acc = torch.tensor(0.85)
f_acc = t_acc.item()
assert isinstance(f_acc, float)
print(f"Q5 ✅  .item() turns a Tensor ({t_acc.dtype}) into a Python float = {f_acc:.4f}")
print("\n🎉 Train/eval loop whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l64_review = {
    "cross_entropy_understood": None,  # understand initial CE ≈ log(num_classes), the expected loss of random guessing? True/False
    "train_epoch_impl":         None,  # train_epoch() correct: the 5-step loop, returns avg_loss as a float? True/False
    "eval_accuracy_impl":       None,  # eval_accuracy() correct: model.eval() + no_grad? True/False
    "confusion_matrix_read":    None,  # can read a confusion matrix, know diagonal = correct predictions? True/False
    "whiteboard_passed":        None,  # all 5 whiteboard derivations done? True/False
}

unfilled = [k for k, v in l64_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l64_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L64 all clear! On to L65: training visualized')

---

→ **Next lesson** · [L65 · Training visualized](L65_visual_training.ipynb)

> Next lesson covers **training visualized**: loss / accuracy curves, gradient norms, weight-distribution histograms.